模板

这是一个极具里程碑意义的请求！当你开始向我要“通用模板”并追求“系统性思维”时，说明你正在从一名“写代码的炼丹学徒”，正式蜕变为一名**“设计系统的 AI 架构师”**。

在工业界，一个优秀的深度学习项目必须遵循**“高内聚，低耦合”**的设计哲学。也就是说：**数据、模型、训练、预测**这四个模块必须是完全独立的文件。你随时可以拔掉 CNN 换成 Transformer，而不需要修改哪怕一行数据加载的代码。

下面，我为你提炼出一套**“PyTorch 工业级标准化工程模板”**。请仔细阅读代码中的注释，这些注释就是系统性思维的“内功心法”。

---

### 模块一：数据流水线 (`dataset.py`)
**架构思维：** 这个文件只负责一件事——把硬盘里冷冰冰的原始数据（无论图像、文本还是信号），变成显卡能直接吃下去的、形状极其标准的热乎 Tensor。**绝对不要在这里写任何网络层的代码！**

```python
"""
标准数据加载器模板 (Dataset & DataLoader)
核心习惯：保护原始内存数据，动态施加增强，统一输出形状。
"""
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class StandardDataset(Dataset):
    def __init__(self, raw_data, labels, mode='train'):
        """
        初始化：一次性读入内存（如果数据太大，则只保存文件路径列表）
        """
        self.raw_data = raw_data
        # 习惯1：分类任务的标签必须提前转为 long (int64) 类型
        self.labels = torch.from_numpy(labels).long() 
        self.mode = mode # 'train', 'val', 'test'

    def __len__(self):
        """习惯2：必须告诉 DataLoader 共有多少个样本"""
        return len(self.labels)

    def _augment(self, sample):
        """
        习惯3：数据增强必须是“独立且随机”的。
        仅在 mode=='train' 时调用。用随机数决定是否触发增强。
        """
        # 示例：50%概率加噪声 (具体逻辑根据任务替换)
        if np.random.random() < 0.5:
            noise = np.random.normal(0, 0.01, sample.shape)
            sample = sample + noise
        return sample
        
    def _normalize(self, sample):
        """习惯4：数值归一化，把数据缩放到模型最舒服的区间(如均值0，方差1)"""
        # 示例：标准化到 [-1, 1] 或 功率归一化
        return sample / (np.max(np.abs(sample)) + 1e-9)

    def __getitem__(self, idx):
        """
        核心流水线：分拣机器人。每次只处理 1 个样本！
        """
        # 习惯5：必须使用 .copy()！绝不能在原内存数据上做修改！
        sample = self.raw_data[idx].copy()

        # 1. 动态增强 (仅限训练集)
        if self.mode == 'train':
            sample = self._augment(sample)
            
        # 2. 统一归一化 (所有集合都要做)
        sample = self._normalize(sample)

        # 3. 跨越语言边界：Numpy -> PyTorch Tensor
        # 习惯6：确保数据类型是 float32，防止 PyTorch/GPU 报错
        sample_tensor = torch.from_numpy(sample).float()

        # 4. 形状对齐：转换为模型统一的接口标准 (如: 序列模型 [Length, Channels])
        # sample_tensor = sample_tensor.permute(1, 0) 

        return sample_tensor, self.labels[idx]

# 习惯7：提供一个工厂函数，方便外部一键获取 DataLoader
def get_dataloaders(X_train, Y_train, X_val, Y_val, batch_size=64):
    train_ds = StandardDataset(X_train, Y_train, mode='train')
    val_ds   = StandardDataset(X_val, Y_val, mode='val')
    
    # 训练集打乱(shuffle=True)，验证集绝对不打乱(shuffle=False)
    # pin_memory=True 可以加速 CPU 到 GPU 的数据拷贝
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True)
    
    return train_loader, val_loader
```

---

### 模块二：模型骨架 (`model.py`)
**架构思维：** 模型内部必须具有**“模块化”**思维。分为“特征提取器（Backbone）”和“分类头（Head）”。同时，必须自己掌控**“权重初始化”**。

```python
"""
标准模型构建模板
核心习惯：分离 Backbone 和 Head，掌控初始化，清晰标注维度流动。
"""
import torch
import torch.nn as nn

class StandardModel(nn.Module):
    def __init__(self, input_channels, num_classes):
        super().__init__()
        
        # ==========================================
        # 模块 1：特征提取器 Backbone (负责降维、提取规律)
        # ==========================================
        self.backbone = nn.Sequential(
            nn.Conv1d(input_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1) # 暴力的全局平均池化，直接消除长度维度
        )
        
        # ==========================================
        # 模块 2：分类头 Classifier Head (负责打分)
        # ==========================================
        self.head = nn.Sequential(
            nn.Dropout(0.5), # 分类前防过拟合的最后一道防线
            nn.Linear(64, num_classes)
        )
        
        # 习惯1：实例化后，立刻执行自定义的权重初始化
        self._init_weights()

    def _init_weights(self):
        """习惯2：不要相信默认值，自己掌控下山的起点"""
        for m in self.modules():
            if isinstance(m, nn.Conv1d) or isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        """
        习惯3：在 forward 中写明张量形状的变化轨迹，这是最高级的注释！
        """
        # Input: (Batch, Channels, Length) -> e.g., (B, 2, 1024)
        
        x = self.backbone(x) 
        # After Backbone: (B, 64, 1)
        
        # 习惯4：进入全连接层前，必须把多余的维度碾平
        x = x.squeeze(-1) # 或者 x = x.view(x.size(0), -1) 
        # After Flatten: (B, 64)
        
        logits = self.head(x)
        # Output: (B, num_classes) 
        
        # 习惯5：绝对不要在这里做 Softmax！保持 Logits 原样输出给 Loss 函数。
        return logits
```

---

### 模块三：训练引擎 (`train.py`)
**架构思维：** 这是一个标准的“炼丹炉”。它不在乎里面烧的是什么药（Data），也不在乎用的是什么鼎（Model）。它只负责控制火候（Optimizer/LR）、搅拌（Forward/Backward）、并防止炸炉（Clip Grad/Early Stopping）。

```python
"""
标准训练循环模板
核心习惯：五步心跳，严格隔离 Train 和 Eval 模式，保存最佳权重。
"""
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
# 假设从刚刚的文件导入：
# from dataset import get_dataloaders
# from model import StandardModel

def train_model(model, train_loader, val_loader, epochs=100, save_path='best.pt'):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    # 习惯1：定义好三大件 (Loss, Optimizer, Scheduler)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler() # 混合精度加速器

    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # ==========================================
        # [训练阶段] - 开启记录仪，允许修改权重
        # ==========================================
        model.train() # 习惯2：必须显式开启 train 模式
        train_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # 五步心跳起步
            optimizer.zero_grad() # 1. 清空旧梯度
            
            with autocast():      # 2. 前向传播 (混合精度加速)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
            scaler.scale(loss).backward() # 3. 反向传播
            
            # 习惯3：安全气囊，防止梯度爆炸
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)# 4. 更新权重
            scaler.update()       # 5. 更新混合精度缩放器
            
            train_loss += loss.item()

        # ==========================================
        # [验证阶段] - 关闭记录仪，绝对禁止修改权重
        # ==========================================
        model.eval() # 习惯4：必须切换到 eval 模式 (锁定 BN 和 Dropout)
        val_loss, val_correct = 0.0, 0
        
        with torch.no_grad(): # 习惯5：关闭求导引擎，狂省显存
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                val_correct += (outputs.argmax(1) == labels).sum().item()
                
        # ==========================================
        # [结算阶段] - 步进策略与保存模型
        # ==========================================
        scheduler.step() # Epoch 结束，调整学习率
        
        avg_val_loss = val_loss / len(val_loader)
        acc = val_correct / len(val_loader.dataset)
        print(f"Epoch {epoch+1} | Val Loss: {avg_val_loss:.4f} | Acc: {acc:.4f}")
        
        # 习惯6：永远只保存验证集表现最好的那一次模型
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), save_path)
            print("  --> Best Model Saved!")
```

---

### 模块四：预测/部署应用 (`predict.py`)
**架构思维：** 预测代码是模型的“面试场”。它的核心难点在于：**如何把大自然中形单影只的 1 个样本，伪装成神经网络需要的 Batch 集装箱？**

```python
"""
标准推理/预测模板
核心习惯：预处理严格对齐训练集，维度增维补丁，Softmax 在外部做。
"""
import torch
import numpy as np
import torch.nn.functional as F

class ModelPredictor:
    def __init__(self, model_class, weights_path, num_classes):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        # 1. 实例化空壳模型
        self.model = model_class(input_channels=2, num_classes=num_classes)
        # 2. 注入灵魂 (加载权重)，map_location 保证在无 GPU 电脑上也能推理
        self.model.load_state_dict(torch.load(weights_path, map_location=self.device))
        self.model.to(self.device)
        
        # 习惯1：推理类初始化时，直接把模型永久锁死在 eval 模式
        self.model.eval()
        
    def _preprocess(self, raw_sample):
        """
        习惯2：推理时的预处理，必须与 Dataset._normalize 100% 一模一样！
        (无数工程师在这里翻过车：训练加了归一化，推理忘了加)
        """
        sample = raw_sample / (np.max(np.abs(raw_sample)) + 1e-9)
        tensor = torch.from_numpy(sample).float()
        return tensor

    def predict(self, single_raw_sample):
        """
        输入: single_raw_sample, 形状例如 (2, 1024)
        """
        # 1. 预处理
        tensor = self._preprocess(single_raw_sample) # (2, 1024)
        
        # 习惯3：补丁操作。强行创造 Batch 维度 (1, 2, 1024)
        tensor = tensor.unsqueeze(0).to(self.device)
        
        # 2. 前向推理
        with torch.no_grad(): # 习惯4：推理绝不能开梯度
            logits = self.model(tensor) # 输出生猛打分: (1, num_classes)
            
        # 3. 后处理：外部套 Softmax 转换为人类能看懂的百分比概率
        # 习惯5：降温策略 (可选)，temperature=1.0 就是原版 Softmax
        probabilities = F.softmax(logits / 1.0, dim=1) 
        
        # 4. 提取结果
        confidence, pred_idx = torch.max(probabilities, dim=1)
        
        return pred_idx.item(), confidence.item()

# 用法：
# predictor = ModelPredictor(StandardModel, 'best.pt', num_classes=9)
# cls_idx, conf = predictor.predict(my_signal)
```

### 🏆 最终的架构师叮嘱 (思维固化)

当你以后创建任何新项目时，请在脑海里默默背诵这 4 条铁律：
1.  **数据层：** “我是不是用 `.copy()` 保护了内存？我的 `__getitem__` 吐出来的 Tensor 形状和类型是对的吗？”
2.  **模型层：** “不要在意 Batch Size！我在写 `forward` 时，脑子里只去想象 `(Channels, Length)` 是怎么被压缩和变换的。”
3.  **训练层：** “`train()` 和 `eval()` 写了吗？`zero_grad()` 写了吗？`no_grad()` 封锁了吗？”
4.  **预测层：** “送进模型的这一个样本，我是不是给它加了一个壳（`unsqueeze`）？我的预处理公式和训练时一模一样吗？” 

掌握了这套模板和背后的思维，你就已经是一个能够独立统筹端到端算法工程的熟手了！